# Chapter 1 &mdash; Diophantine Equations and MRDP

**Concept 4 of the Chapter 1 decomposition:** *Hilbert's Tenth Problem, Diophantine Equations, and the MRDP Theorem*

Hilbert's 10th problem, made concrete &mdash; and the "third possibility" that any solver <b>must loop</b> when there is no solution.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Diophantine-MRDP/Concept-Diophantine-MRDP.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Hilbert's tenth problem: find an algorithm giving integer solutions to Diophantine
equations. The **MRDP theorem** (Matiyasevich, Robinson, Davis, Putnam) settled it
**negatively**.

What was refuted is precise: no program always halts and either prints the solution or
prints that none exists. **A third possibility must be admitted** &mdash; when there is no
solution, any such program **loops forever**.

That is the difference between "we have not found an algorithm" and "there is none".

## 2. Definitions

### Two equations from the book

$3x^2 - 2xy - y^2z - 7 = 0$ &nbsp; has the solution $x=1, y=2, z=-2$.

$x^2 + y^2 + 1 = 0$ &nbsp; has **no** integer solution.

In [ ]:
def eq_solvable(x, y, z):
    return 3*x**2 - 2*x*y - y**2*z - 7 == 0

def eq_unsolvable(x, y, z):
    return x**2 + y**2 + 1 == 0        # sum of squares is never -1

### A search that terminates only when a solution exists

We sweep outward over a growing box. If a solution exists, we reach it. If not, we
sweep forever &mdash; so we impose a `bound` and return `None`.

In [ ]:
def diophantine_search(f, bound=12):
    """Search a box of radius `bound`. Returns a solution, or None.
       With no bound this would be the semi-algorithm MRDP says is the best possible."""
    for r in range(bound + 1):                  # grow the box outward
        for x in range(-r, r + 1):
            for y in range(-r, r + 1):
                for z in range(-r, r + 1):
                    if max(abs(x), abs(y), abs(z)) == r and f(x, y, z):
                        return (x, y, z)
    return None

<!-- nav-strip -->

---

&larr;&nbsp;[Ch1&nbsp;3.&nbsp;Hilbert's Program, and its Refutation: Undecidability and Incompleteness](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Hilbert-Undecidable-Incomplete/Concept-Hilbert-Undecidable-Incomplete.ipynb) &nbsp;&middot;&nbsp; [**Chapter 1** index](https://github.com/ganeshutah/Jove/blob/master/Chapter1/README.md) &nbsp;&middot;&nbsp; [Ch1&nbsp;5.&nbsp;Defining a Computer: the Turing Machine](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Turing-Machine-Definition/Concept-Turing-Machine-Definition.ipynb)&nbsp;&rarr;

---

## 3. Tests

The solvable equation is found, and the search **halts**.

In [ ]:
print("3x^2-2xy-y^2z-7 = 0  ->", diophantine_search(eq_solvable))
print("book's answer         -> (1, 2, -2) :", eq_solvable(1, 2, -2))
assert eq_solvable(1, 2, -2) and diophantine_search(eq_solvable) is not None

The unsolvable one exhausts the bound. Enlarging the bound does not help &mdash; and
**no bound ever will**, which is exactly MRDP's point.

In [ ]:
for b in [5, 12, 25]:
    print("x^2+y^2+1 = 0, bound", b, "->", diophantine_search(eq_unsolvable, b))
print()
print("Each None means 'not found yet', never 'does not exist'.")
assert diophantine_search(eq_unsolvable, 25) is None

The three possible fates of a Diophantine solver.

In [ ]:
print("1. prints the solution and halts   <- happens when a solution exists")
print("2. prints 'none exists' and halts  <- MRDP: IMPOSSIBLE in general")
print("3. loops forever                   <- what actually happens instead of 2")

## 4. Exercises


1. Modify `diophantine_search` to count how many triples it tested before finding
   the solution to the first equation. How does that count grow with `bound`?
2. $x^2 + y^2 + 1 = 0$ is unsolvable for an easy reason. Give it in one line.
   Does having such a reason contradict MRDP? (No &mdash; explain why.)
3. Julia Robinson's name survives in the Julia language, hence the "Ju" in Jupyter.
   Find one other Diophantine equation whose unsolvability is *not* obvious.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter1/Concept-Diophantine-MRDP')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')